In [1]:
# --- PHASE 1: Environment & Config ---
import os, sys, gc, random, time, torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

# Root Setup
root = Path('/home/jupyter-1nt23cb058/Capstone')
os.chdir(root)
if str(root) not in sys.path: sys.path.insert(0, str(root))

# Custom Model Imports
from gnn.model import STPIGNN, LossBreakdown
import gnn.model as gnn_model
import shared.physics_config as phys_cfg

# Global Settings
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training Hyperparameters
CFG = {
    'lr': 1e-5, 
    'max_epochs': 15, 
    'train_stride': 128, 
    'val_stride': 24,
    'total_nodes': 154902
}

def save_state(path, epoch, step, best_val):
    payload = {
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler_amp.state_dict(),
        'epoch': int(epoch), 'step': int(step), 'best_val_mse': float(best_val),
        'timestamp': time.ctime()
    }
    torch.save(payload, path)

print(f"✅ Environment ready on {device}. Root: {root}")

✅ Environment ready on cuda. Root: /home/jupyter-1nt23cb058/Capstone


In [2]:
# --- PHASE 3: Spatial Partitioning ---
from sklearn.cluster import KMeans

# 1. Load Global Topology
pyg = torch.load(root / 'data/processed/graph/topology_graph_pyg_inference.pt', weights_only=False)
edge_index_global = pyg.edge_index.long().cpu()
edge_attr_global = pyg.edge_attr.float().cpu()
num_nodes = int(pyg.num_nodes)
train_mask_global = pyg.train_mask.bool().cpu()

# 2. Load Coordinates
node_map = pd.read_parquet(root / 'data/processed/graph/topology_nodeid_to_index_map.parquet')
nodes_df = pd.read_parquet(root / 'data/graphs/bangalore_utm_nodes.parquet')
coords_df = node_map.merge(nodes_df[['osmid', 'x', 'y']], left_on='node_id', right_on='osmid', how='inner').sort_values('node_index')
coords = coords_df[['x', 'y']].to_numpy(dtype='float32')

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask, upwind_mask):
        self.cid, self.n_id = cid, torch.as_tensor(node_ids, dtype=torch.long)
        self.edge_index, self.edge_attr = edge_index.long(), edge_attr.float()
        self.train_mask, self.upwind_edge_mask = train_mask.bool(), upwind_mask.bool()
        self.x = None 

# 3. Partition & Relativize Shield
print(f"Partitioning {num_nodes} nodes into 64 clusters...")
kmeans = KMeans(n_clusters=64, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(coords)
cluster_data = []
edge_index_np = edge_index_global.numpy()

for cid in range(64):
    n_ids = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(n_ids) == 0: continue
    
    keep = np.isin(edge_index_np[0], n_ids) & np.isin(edge_index_np[1], n_ids)
    local_map = {g: i for i, g in enumerate(n_ids.tolist())}
    re_src = np.array([local_map[int(x)] for x in edge_index_np[0, keep]], dtype=np.int64)
    re_dst = np.array([local_map[int(x)] for x in edge_index_np[1, keep]], dtype=np.int64)
    
    cluster_data.append(SpatialPartition(
        cid, n_ids, torch.tensor(np.stack([re_src, re_dst]), dtype=torch.long),
        torch.tensor(edge_attr_global[keep], dtype=torch.float32),
        train_mask_global[torch.as_tensor(n_ids)],
        torch.zeros(re_src.shape[0], dtype=torch.bool)
    ))

print(f"Phase 3 Complete: {len(cluster_data)} clusters hard-aligned.")

Partitioning 154902 nodes into 64 clusters...


/tmp/ipykernel_330074/3766077896.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_attr_global[keep], dtype=torch.float32),


Phase 3 Complete: 64 clusters hard-aligned.


In [3]:
# --- PHASE 3: Spatial Partitioning & Topology Reconstruction ---
# Purpose: Load global graph, partition into 64 clusters, and REPAIR indices to prevent CUDA OOM.
from sklearn.cluster import KMeans

# 1. Load Global Topology
pyg = torch.load(root / 'data/processed/graph/topology_graph_pyg_inference.pt', weights_only=False)
edge_index_global = pyg.edge_index.long().cpu()
edge_attr_global = pyg.edge_attr.float().cpu() if hasattr(pyg, 'edge_attr') else torch.ones(edge_index_global.shape[1], 1)
num_nodes = int(pyg.num_nodes)
train_mask_global = pyg.train_mask.bool().cpu()

# 2. Load Coordinates for K-Means
node_map_df = pd.read_parquet(root / 'data/processed/graph/topology_nodeid_to_index_map.parquet')
nodes_df = pd.read_parquet(root / 'data/graphs/bangalore_utm_nodes.parquet')
coords_df = node_map_df.merge(nodes_df[['osmid', 'x', 'y']], left_on='node_id', right_on='osmid', how='inner').sort_values('node_index')
coords = coords_df[['x', 'y']].to_numpy(dtype='float32')

# 3. Partition and Relativize
print(f"Partitioning {num_nodes} nodes into 64 clusters...")
kmeans = KMeans(n_clusters=64, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(coords)

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask, upwind_mask):
        self.cid = cid
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long)
        self.edge_index = edge_index.long()
        self.edge_attr = edge_attr.float()
        self.train_mask = train_mask.bool()
        self.upwind_edge_mask = upwind_mask.bool()
        self.x = None # To be filled in Phase 4

cluster_data = []
edge_index_np = edge_index_global.numpy()
for cid in range(64):
    node_ids = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(node_ids) == 0: continue
    
    # RELATIVIZATION SHIELD: Ensure indices are 0 to N-1 for THIS cluster
    keep = np.isin(edge_index_np[0], node_ids) & np.isin(edge_index_np[1], node_ids)
    local_edge_global = edge_index_np[:, keep]
    local_map = {g: i for i, g in enumerate(node_ids.tolist())}
    re_src = np.array([local_map[int(x)] for x in local_edge_global[0]], dtype=np.int64)
    re_dst = np.array([local_map[int(x)] for x in local_edge_global[1]], dtype=np.int64)
    
    cluster_data.append(SpatialPartition(
        cid, node_ids, torch.tensor(np.stack([re_src, re_dst]), dtype=torch.long),
        torch.tensor(edge_attr_global[keep], dtype=torch.float32),
        train_mask_global[torch.as_tensor(node_ids)],
        torch.zeros(local_edge_global.shape[1], dtype=torch.bool) # Placeholder for upwind
    ))

print(f"Phase 3 Complete: {len(cluster_data)} clusters relativized and safe for GPU.")

Partitioning 154902 nodes into 64 clusters...


/tmp/ipykernel_330074/878035478.py:48: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_attr_global[keep], dtype=torch.float32),


Phase 3 Complete: 64 clusters relativized and safe for GPU.


In [4]:
# --- PHASE 4: Manifold Injection ---
master_df = pd.read_parquet('data/processed/graph/master_scaled_checkpoint.parquet')
feature_cols = ['pm2_5_scaled', 'pm10_scaled', 'nitrogen_dioxide_scaled', 'sulphur_dioxide_scaled', 
                'carbon_monoxide_scaled', 'wind_speed_10m_scaled', 'wind_direction_10m_scaled', 
                'wind_gusts_10m_scaled', 'temperature_2m_scaled', 'relative_humidity_2m_scaled', 'surface_pressure_scaled']

# 1. Map to Dictionary for speed
data_dict = {}
for node_val, group in tqdm(master_df.groupby('node_index'), desc="Indexing Nodes"):
    vals = group.sort_values('time').tail(12)[feature_cols].values
    if vals.shape[0] > 0:
        t_vals = torch.zeros((12, 11))
        t_vals[:vals.shape[0], :] = torch.from_numpy(vals).float()
        data_dict[node_val] = t_vals

# 2. Populate x
for cluster in tqdm(cluster_data, desc="Injecting Manifold"):
    feat_matrix = torch.zeros((len(cluster.n_id), 12, 16))
    for i, g_id in enumerate(cluster.n_id.tolist()):
        if g_id in data_dict: feat_matrix[i, :, :11] = data_dict[g_id]
    cluster.x = feat_matrix

del master_df, data_dict; gc.collect()
print("Phase 4 Complete: Feature Manifold Populated.")

Indexing Nodes:   0%|          | 0/17 [00:00<?, ?it/s]

Injecting Manifold:   0%|          | 0/64 [00:00<?, ?it/s]

Phase 4 Complete: Feature Manifold Populated.


In [5]:
# --- PHASE 5 & 6: DataLoaders & Model Initialization ---

class LazyClusterDataset(Dataset):
    def __init__(self, clusters, t0, t1, stride):
        self.clusters, self.starts = clusters, list(range(t0, t1 - 12, stride))
    def __len__(self): return len(self.starts) * len(self.clusters)
    def __getitem__(self, idx):
        return torch.zeros(1), torch.zeros(1), idx % len(self.clusters)

train_loader = DataLoader(LazyClusterDataset(cluster_data, 0, 87178, CFG['train_stride']), batch_size=1, shuffle=True)
val_loader = DataLoader(LazyClusterDataset(cluster_data, 87178, 87514, CFG['val_stride']), batch_size=1)

model = STPIGNN(node_in_dim=16, edge_dim=edge_attr_global.shape[-1], 
                spatial_hidden_dim=96, temporal_hidden_dim=96, gnn_layers=2).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-2)
scaler_amp = torch.amp.GradScaler('cuda')

print(f"Phase 5/6 Complete: Model initialized at LR {CFG['lr']}")

Phase 5/6 Complete: Model initialized at LR 1e-05


In [6]:
# --- PHASE 7: Training Engine ---
checkpoint_path, autosave_path, best_path = 'stable.pt', 'autosave.pt', 'best.pt'
start_epoch, start_step, best_val_mse = 1, 0, float('inf')
last_save_time = time.time()

# Corruption-Resistant Load
for path in [checkpoint_path, autosave_path, best_path]:
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location=device)
            model.load_state_dict(ckpt['state_dict']); optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch, start_step, best_val_mse = ckpt['epoch'], ckpt['step'], ckpt['best_val_mse']
            print(f"✅ Resumed from {path} at Epoch {start_epoch}")
            break
        except: continue

try:
    for epoch in range(start_epoch, 16):
        model.train()
        curr_lambda = float(phys_cfg.PHYSICS_LOSS_LAMBDA) * min(1.0, epoch / 12.0)
        pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch}")
        
        for i, (_, yb_raw, c_idx_t) in enumerate(train_loader):
            if epoch == start_epoch and i < start_step:
                pbar.update(1); continue

            part = cluster_data[int(c_idx_t[0].item())]
            if part.edge_index.numel() > 0 and part.edge_index.max() >= part.x.size(0): continue

            xb = part.x.unsqueeze(0).to(device, non_blocking=True)
            edge_i, edge_a = part.edge_index.to(device), part.edge_attr.to(device)
            
            data_in = (part.x[:, 0, :5].sum(dim=-1) > 0).to(device)
            mask = part.train_mask.to(device) | data_in
            u_mask = part.upwind_edge_mask.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda'):
                pred = model(xb, edge_i, edge_a)
                if mask.any():
                    res = gnn_model.compute_total_loss(pred, yb_raw.to(device), mask, edge_i, edge_a, u_mask, curr_lambda)
                    mode = 'STATION'
                else:
                    phys_p = gnn_model.physics_upwind_penalty(pred, edge_i, u_mask, edge_a)
                    res = LossBreakdown(total=curr_lambda * phys_p, data=pred.new_tensor(0.0), physics=phys_p)
                    mode = 'PHYS'

            if torch.isfinite(res.total):
                scaler_amp.scale(res.total).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                scaler_amp.step(optimizer); scaler_amp.update()

            pbar.set_postfix({'M': mode, 'D': f'{res.data.item():.4f}', 'City%': f'{(i/len(train_loader))*100:.1f}%'})
            pbar.update(1)

            if i % 50 == 0: torch.cuda.empty_cache()
            if (time.time() - last_save_time) > 1800:
                save_state(autosave_path, epoch, i, best_val_mse); last_save_time = time.time()

        pbar.close()
        save_state(checkpoint_path, epoch + 1, 0, best_val_mse)

except KeyboardInterrupt:
    print("\nEmergency Save..."); save_state(checkpoint_path, epoch, i, best_val_mse)
except Exception as e: print(f"FAIL: {e}")

Epoch 1:   0%|          | 0/43584 [00:00<?, ?it/s]

FAIL: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [149,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [149,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [149,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [149,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [149,0,0], thread: [4,0,0] Asser